In [ ]:
#| default_exp logger

# logger

> simple logger using idiomatic Solveit

In [ ]:
#| export
import time
from fastcore.all import patch
from datetime import datetime
from anyio import sleep
from anyio.from_thread import start_blocking_portal
from dialoghelper.core import update_msg, add_msg, del_msg, read_msg, find_msg_id, msg_idx, find_dname

In [ ]:
import random
import fastcore.all as FC
from fastcore.test import *
from dutil.core import waitpred

In [ ]:
#| export
_notfound = {'msg': {}}
def findlogtarget(id:str='', n:int=1):
    if (msg := read_msg(0, id=id) if id else read_msg(n)) == _notfound: return ''
    return msg.id if (cnt := msg.content).count('\n') == cnt.count('\u200b')-1 else ''

In [ ]:
test_eq(findlogtarget(read_msg().id), read_msg().id)

In [ ]:
test_eq(findlogtarget(), read_msg(1).id)

In [ ]:
test_eq(findlogtarget(n=-1), read_msg().id)

In [ ]:
#| export
class Logger:
    logs:list; msgid:str; _s:str
    def __init__(self, id:str|None=''): self.logs = []; self._s='\u200b'; self.setup(id, True)
    def setup(self, id:str|None='', clear:bool=False): 
        self.msgid = id
        if id is not None:
            if id and isinstance(msg_idx(id), int):  # reuse given target
                if not clear: update_msg(self.msgid, msg_type='raw')
            elif id := findlogtarget():  # reuse or create a new target
                self.msgid = id
                if clear: self.clear()
            else: self.msgid = add_msg('\u200b' if clear else self._s, msg_type='raw'); clear = False
        if clear: self.clear()
    def clear(self): 
        self.logs.clear(); self._s='\u200b'; 
        if self.msgid: update_msg(self.msgid, content=self._s, msg_type='raw')
    def settle(self, to:float=2.0, interval:float=0.1):
        if logid := self.msgid:
            dname, thisloc = find_dname(), msg_idx()  # capture dname, find_dname is not threadsafe
            async def _wait():
                t0 = time.time()
                while time.time()-t0 < to and msg_idx(logid, dname=dname) != thisloc: await sleep(interval)
                return msg_idx(logid, dname=dname) == thisloc
            with start_blocking_portal() as portal: return portal.call(_wait)
    def __call__(self, msg, *args, **kwargs): 
        dt = datetime.now()
        s = f"[{dt:%H:%M:%S}.{dt.microsecond//1000:03d}] {msg}"
        self.logs.insert(0, s)
        if self.msgid:
            self._s = f"\u200b{s}" + (f"\n{self._s}" if self._s != '\u200b' else '')
            update_msg(self.msgid, content=self._s)
        else: print(s, flush=True)


In [ ]:
log = Logger()

In [ ]:
log('test')
log('test2')
log('test3')

In [ ]:
log(s := ';qwedcv fjkds')
test_is(s in log.logs[0], True)
log.logs

['[23:52:03.592] ;qwedcv fjkds',
 '[23:51:48.820] test3',
 '[23:51:48.796] test2',
 '[23:51:48.770] test']

In [ ]:
log.clear()
test_eq(log.logs, [])

In [ ]:
log(f"Some msg {random.randint(0, 1000)}")
log.logs

['[23:52:23.831] Some msg 85']

In [ ]:
#| export
@patch
def show(self:Logger, clear:bool=False, replace:bool=False):
    if not self.msgid: return
    if  (loc := msg_idx(self.msgid)) and msg_idx() == loc-1: return
    oldid = self.msgid
    self.setup(find_msg_id() if replace else findlogtarget(), clear)
    if oldid != self.msgid: del_msg(oldid)
    time.sleep(0.1)
    self.settle()
    if self.msgid and not clear: update_msg(id=self.msgid, content=self._s)

In [ ]:
if (msgid := findlogtarget()) and msgid != log.msgid: del_msg(msgid)
log.show()

In [ ]:
add_msg(content='\u200b[09:53:00] some other msg\n\u200b[09:44:16] some msg\n', id=read_msg(1).id, msg_type='raw');

In [ ]:
log.show()

In [ ]:
log.show()

In [ ]:
# log.show(replace=True)  # duplicate, uncomment and run 

In [ ]:
for _ in "En algún lugar de la Mancha".split(): log(_)

In [ ]:
log.show()

Look at `show` method. What it does?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The `show` method repositions the logger's output message in the dialog. Here's what it does:

1. **Checks current position** - If the logger's message is already right after the current message, it does nothing
2. **Finds new location** - Either replaces the current message (if `replace=True`) or finds/creates a log target message nearby
3. **Moves the log** - Updates `self.msgid` to the new location and deletes the old message if it changed
4. **Waits to settle** - Gives the UI time to update and waits for the message to stabilize at the expected position
5. **Updates content** - Refreshes the message content (unless `clear=True`, which would clear it)

Essentially, it lets you move the log output to appear near where you're currently working in the dialog, which is useful when you've scrolled away from where the logger was initially created.

# export -

In [ ]:
from dutil.flakes import show_flakes
show_flakes()

No warnings to report

In [ ]:
# #|hide
# #|eval: false
# import fastcore.all as FC
# from nbdev import nbdev_export
# if FC.IN_NOTEBOOK:
#     nb_path = '01_logger.ipynb'
#     nbdev_export(nb_path)